# Validação cruzada

**Objetivo:** comparar uma única divisão treino/teste com a validação cruzada k-fold e ver por que a média do CV é mais confiável.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Paleta do curso (idêntica ao site)
INK, PAPER = "#1a1a1a", "#fffdf8"
BLUE, RED, GREEN, MUTED = "#3266ad", "#c0392b", "#1a7a4a", "#6b6457"

plt.rcParams.update({
    "font.family": "serif", "font.size": 12,
    "figure.facecolor": PAPER, "axes.facecolor": PAPER,
    "axes.edgecolor": "#b9ad95", "axes.grid": True,
    "grid.color": "#e2d9c4", "grid.linewidth": 0.7,
    "axes.labelcolor": INK, "text.color": INK,
    "xtick.color": MUTED, "ytick.color": MUTED,
})
rng = np.random.default_rng(42)

In [ ]:
from sklearn.datasets import load_iris
X, y = load_iris(return_X_y=True)

## Uma divisão só varia bastante

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier

for s in range(5):
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=s)
    acc = KNeighborsClassifier().fit(Xtr, ytr).score(Xte, yte)
    print(f'seed={s} -> acurácia={acc:.3f}')

## k-fold: estimativa estável (com pipeline, sem vazamento)

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

modelo = make_pipeline(StandardScaler(), KNeighborsClassifier())
scores = cross_val_score(modelo, X, y, cv=5)
print('folds:', np.round(scores, 3))
print(f'CV: {scores.mean():.3f} ± {scores.std():.3f}')

## Exercícios

**1.** Rode a validação cruzada com cv=10. A média muda muito? E o desvio?

**2.** Use `StratifiedKFold` e confirme que cada fold preserva a proporção das classes.

In [ ]:
# @title Solução (clique para revelar)
s10 = cross_val_score(modelo, X, y, cv=10)
print(f'cv=10: {s10.mean():.3f} ± {s10.std():.3f}')

from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold(n_splits=5)
for i, (_, te) in enumerate(skf.split(X, y)):
    vals, cnt = np.unique(y[te], return_counts=True)
    print(f'fold {i}: {dict(zip(vals, cnt))}')